# Conditional Strategies II: Timing Expected Returns

The ideas are in the slides. Here you test the most famous timing signal for the market, the dividend yield: in sample,
then the honest way, out of sample, and then as a strategy against buy and hold. Then the same three steps for four
factors, each with its own price signal, the value spread. Then a search over 24 more signals, and how many survive.
Last, the other market signals from the slides, one at a time.

**The plan**, one section per exercise in the slides

1. The dividend yield, at three horizons
2. Out of sample
3. The timing strategy, against buy and hold
4. 🎯 Each factor's value spread: your turn
5. Timing the six factors: 24 tries
6. Pick a predictor: any of the signals from the slides. Then back to the slides
7. 🎯 Challenge (homework): one number and three sentences

> 🧩 **Every section stands alone.** Run the Setup cell once, then jump to any section, in any order. Within a section,
> run the cells from the top.

📑 **Slides for this class:** [Conditional Strategies II (PDF)](https://amoreira2.github.io/UG54/slides/conditional_strategies_2.pdf)

<iframe src="../../slides/conditional_strategies_2.pdf" width="100%" height="520" style="border: 1px solid #ddd;"></iframe>

In [ ]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = [11, 4]
import warnings; warnings.filterwarnings('ignore')

BASE = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"

# ── the market: its return, the T-bill rate and the dividend yield, 1926-2024
crsp  = pd.read_csv(f"{BASE}/Markettiming_data1.csv", parse_dates=['date'], index_col='date')
crsp.index = crsp.index + pd.offsets.MonthEnd(0)
tbill = pd.read_parquet(f"{BASE}/factors_monthly_1926.parquet")[['RF']]

market = crsp.merge(tbill, on='date').dropna()
market['ret']          = market['vwretd'] - market['RF']                        # the market's excess return
market['log_ret']      = np.log1p(market['vwretd']) - np.log1p(market['RF'])    # the same in logs: these add up across months
market['dp']           = market['dp'].rolling(12).sum()                         # dividend yield: last 12 months of dividends over today's price
market['next_ret']     = market['ret'].shift(-1)                                # NEXT month's return, on this month's row
market['next_log_ret'] = market['log_ret'].shift(-1)
market = market[['dp', 'ret', 'log_ret', 'next_ret', 'next_log_ret']]

# ── the factors
four_factors = pd.read_parquet(f"{BASE}/value_spreads_monthly_1926.parquet")    # HML, SMB, RMW, CMA, and the book-to-market of their legs
six_factors  = pd.read_csv(f"{BASE}/ff_monthly.csv", parse_dates=['date'], index_col='date')
six_factors  = six_factors[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'UMD']].dropna()   # monthly returns, 1980-2026

# ── the other market signals from the slides, for section 6: Goyal and Welch's data, updated to 2025, and VIX
goyal = pd.read_csv(f"{BASE}/goyal_welch_monthly_2025.csv", parse_dates=['date'], index_col='date')
vix   = pd.read_csv(f"{BASE}/vix_monthly_fred.csv", parse_dates=['date'], index_col='date')   # last close of each month

price_level       = (1 + goyal['infl']).cumprod()                               # consumer prices, from monthly inflation
real_earnings_10y = (goyal['E12'] / price_level).rolling(120).mean()            # ten years of S&P 500 earnings, in constant dollars

goyal['E/P']               = goyal['E12'] / goyal['Index']                      # S&P 500 earnings over the last 12 months, over price
goyal['1/CAPE']            = real_earnings_10y / (goyal['Index'] / price_level) # Shiller's CAPE, turned into a yield like D/P
goyal['book-to-market']    = goyal['b/m']                                       # Dow Jones Industrials: book value over market value
goyal['short rate']        = goyal['tbl']                                       # 3-month T-bill yield
goyal['term spread']       = goyal['lty'] - goyal['tbl']                        # long-term government bond yield minus the T-bill
goyal['default spread']    = goyal['BAA'] - goyal['AAA']                        # Baa minus Aaa corporate bond yields
goyal['realized variance'] = goyal['svar']                                      # sum of squared daily S&P 500 returns in the month
goyal['net issuance']      = goyal['ntis']                                      # net equity issued by NYSE firms over 12 months, over their value
goyal = goyal[['E/P', '1/CAPE', 'book-to-market', 'short rate', 'term spread', 'default spread',
               'realized variance', 'net issuance']]

predictors = market[['dp']].rename(columns={'dp': 'D/P'})                       # one row per month of the market
predictors = predictors.merge(goyal, on='date', how='left')
predictors = predictors.merge(vix.rename(columns={'vix': 'VIX'}), on='date', how='left')
predictors = predictors.loc['1927':]


def sharpe(ret):
    # Sharpe ratio a year, from monthly excess returns
    return ret.mean() / ret.std() * np.sqrt(12)


def out_of_sample(data, signal, start):
    # The loop of section 2, for any signal. `data` has the signal and `next_ret`, next month's return, on the same row.
    # Each month from `start`: fit next month's return on the signal using the past only, then forecast.
    data = data[[signal, 'next_ret']].dropna()
    rows = []
    for t in data.loc[start:].index:
        past = data.loc[:t].iloc[:-1]                         # the months whose next return was already known at t
        fit  = sm.OLS(past['next_ret'], sm.add_constant(past[signal])).fit()
        rows.append({'date':     t,
                     'ret':      data.loc[t, 'next_ret'],                                          # what happened next month
                     'forecast': fit.params['const'] + fit.params[signal] * data.loc[t, signal],   # what the signal said
                     'average':  past['next_ret'].mean()})                                         # what the historical average said
    return pd.DataFrame(rows).set_index('date')


def r2_out_of_sample(forecasts):
    # 1 - (squared errors of the signal's forecast) / (squared errors of the historical average)
    error_signal  = forecasts['ret'] - forecasts['forecast']
    error_average = forecasts['ret'] - forecasts['average']
    return 1 - (error_signal ** 2).sum() / (error_average ** 2).sum()


print(f"market {market.index[0]:%Y-%m} to {market.index[-1]:%Y-%m}; four factors {four_factors.index[0]:%Y-%m} to "
      f"{four_factors.index[-1]:%Y-%m}; six factors {six_factors.index[0]:%Y-%m} to {six_factors.index[-1]:%Y-%m}; "
      f"{predictors.shape[1]} market signals to {predictors.index[-1]:%Y-%m}")

**What Setup gives you.** Every table has one row per month, and every return is over T-bills.

| name | what is in it |
|---|---|
| `market` | `dp`: the dividend yield. `ret`: the market's return this month. `next_ret`: the market's return **next** month, on this month's row. `log_ret`, `next_log_ret`: the same two in logs |
| `four_factors` | the returns of `HML`, `SMB`, `RMW` and `CMA`, and the book-to-market of each one's long and short legs |
| `six_factors` | the returns of the six factors, 1980–2026 |
| `predictors` | ten market signals, one column each, on the same months as `market` |
| `out_of_sample(data, signal, start)` | the loop of section 2: one forecast a month, each fitted on the past only |
| `r2_out_of_sample(forecasts)` | the out-of-sample $R^2$ of those forecasts |
| `sharpe(ret)` | the Sharpe ratio a year of a monthly return |

A signal known this month has to forecast the return of next month. So every regression below puts this month's signal
and `next_ret` on the same row.

---

## 1. The dividend yield, at three horizons

> 🧩 **Stands alone.** Needs only the Setup cell: you can start here.

When prices are low relative to dividends, either dividends are about to fall, or future returns are high. First compare
the dividend yield with the return over the *next* five years.

In [ ]:
next_5_years = market['log_ret'].rolling(60).sum().shift(-60) / 5      # the return over the next 60 months, a year

ax = market['dp'].plot(label='dividend yield', legend=True)
next_5_years.plot(ax=ax, secondary_y=True, label='return over the next five years, a year', legend=True)
plt.show()

print(f"correlation: {market['dp'].corr(next_5_years):.2f}")

Now the forecasting regression,

$$r_{t \to t+h} = a + b \cdot \text{D/P}_t + u,$$

run for the next month, the next year and the next five years. Two sets of $t$-statistics: the usual ones, and HAC ones
that allow for the overlap.

In [ ]:
table = {}
for months in [1, 12, 60]:
    data = market[['dp']].copy()
    data['next_ret'] = market['log_ret'].rolling(months).sum().shift(-months) * 12 / months   # the return over the next `months` months, a year
    data = data.dropna()

    X = sm.add_constant(data['dp'])
    usual = sm.OLS(data['next_ret'], X).fit()                                                 # as if every month were new
    hac   = sm.OLS(data['next_ret'], X).fit(cov_type='HAC', cov_kwds={'maxlags': months})     # allows for the overlap

    table[f"next {months} months"] = {'b': usual.params['dp'], 'usual t': usual.tvalues['dp'], 'HAC t': hac.tvalues['dp'],
                                      'R2': usual.rsquared, 'observations': len(data), 'do not overlap': len(data) // months}
print(pd.DataFrame.from_dict(table, orient='index').round(3).to_string())

- When was the dividend yield highest? What came next?
- The $R^2$ and the usual $t$ both grow with the horizon. Should you believe the five-year regression most?
- Two consecutive five-year returns share 59 of their 60 months. How many truly separate five-year periods are there?
  Which $t$-statistic would you report?

---

## 2. Out of sample

> 🧩 **Stands alone.** Needs only the Setup cell: you can start here.

The honest test: stand in each month $t$, using only what you knew then.

- fit the regression of next month's return on D/P, using only months whose next return you have already seen
- forecast next month's return with today's D/P
- the benchmark: the historical average return up to $t$, which is what you would use with no signal

The loop below does this every month from 1950. Read it before you run it.

In [ ]:
data = market[['dp', 'next_ret']].dropna()                # this month's D/P and next month's return, on the same row

rows = []
for t in data.loc['1950':].index:
    past = data.loc[:t].iloc[:-1]                         # the months whose next return was already known at t
    fit  = sm.OLS(past['next_ret'], sm.add_constant(past['dp'])).fit()
    rows.append({'date':     t,
                 'ret':      data.loc[t, 'next_ret'],                                       # what happened next month
                 'forecast': fit.params['const'] + fit.params['dp'] * data.loc[t, 'dp'],    # what D/P said
                 'average':  past['next_ret'].mean()})                                      # what the historical average said
forecasts = pd.DataFrame(rows).set_index('date')
forecasts.head()

The out-of-sample $R^2$ compares the squared errors of the two forecasts:

$$R^2_{\text{OOS}} = 1 - \frac{\sum (r - \text{forecast})^2}{\sum (r - \text{average})^2}$$

Positive: D/P beat the historical average. Negative: you would have done better without it.

In [ ]:
error_dp      = forecasts['ret'] - forecasts['forecast']      # how far off D/P's forecast was
error_average = forecasts['ret'] - forecasts['average']       # how far off the historical average was

r2_oos = 1 - (error_dp ** 2).sum() / (error_average ** 2).sum()
print(f"out-of-sample R2, 1950-2024: {r2_oos:.2%}")

for first, last in [('1950', '1979'), ('1980', '1999'), ('2000', '2024')]:
    r2_period = 1 - (error_dp.loc[first:last] ** 2).sum() / (error_average.loc[first:last] ** 2).sum()
    print(f"   {first}-{last}: {r2_period:.2%}")

- In which of the three periods did D/P beat the historical average?
- In sample, D/P explained 0.3% of next month's return. Out of sample it explains less than nothing. How can both be true?
- Why is the benchmark the historical average, and not a forecast of zero?

Setup has these two steps as functions, `out_of_sample()` and `r2_out_of_sample()`. They are the same lines, and the
sections below use them.

---

## 3. The timing strategy, against buy and hold

> 🧩 **Stands alone.** Needs only the Setup cell: you can start here.

The formula from *Capital Allocation I* with a forecast in it: weight $= \text{forecast} / (\gamma \sigma^{2})$. We hold
$\sigma$ fixed at its value before 1950, so all the timing comes from the forecast.

What do you compare it with? Buy and hold **with the same average weight**. Then the two strategies hold the same amount
of the market on average, and the only difference is *when*.

In [ ]:
forecasts = out_of_sample(market, 'dp', '1950')           # section 2's forecasts

gamma    = 3
variance = market.loc[:'1949', 'ret'].var()               # the market's monthly variance before 1950, held fixed

forecasts['weight']       = forecasts['forecast'] / (gamma * variance)       # the weight on the market: negative means short
forecasts['timing']       = forecasts['weight'] * forecasts['ret']           # next month's return on that weight
forecasts['buy and hold'] = forecasts['weight'].mean() * forecasts['ret']    # the same average weight, never changed

print(f"Sharpe ratio:      timing {sharpe(forecasts['timing']):.2f}, buy and hold {sharpe(forecasts['buy and hold']):.2f}")
print(f"volatility a year: timing {forecasts['timing'].std() * np.sqrt(12):.1%}, buy and hold {forecasts['buy and hold'].std() * np.sqrt(12):.1%}")
print(f"average weight {forecasts['weight'].mean():.2f}, short in {np.mean(forecasts['weight'] < 0):.0%} of months\n")
for first, last in [('1950', '1979'), ('1980', '1999'), ('2000', '2024')]:
    period = forecasts.loc[first:last]
    print(f"{first}-{last}: timing {sharpe(period['timing']):.2f}, buy and hold {sharpe(period['buy and hold']):.2f}")

(1 + forecasts[['timing', 'buy and hold']]).cumprod().plot(logy=True, title='cumulative return: timing on D/P, and buy and hold with the same average weight')
plt.show()
forecasts['weight'].plot(title='weight on the market'); plt.axhline(0, color='k', lw=0.5); plt.show()

- When does timing lose ground to buy and hold? What was D/P doing then?
- Timing ends above buy and hold, yet its Sharpe ratio is lower. How can both be true? Look at the two volatilities.
- In which period did timing win, and in which did it lose badly?
- Would changing `gamma` change the Sharpe ratio? Try it.
- Why compare with buy and hold at the same average weight, and not with one dollar in the market?

---

## 4. 🎯 Each factor's value spread

> 🧩 **Stands alone.** Needs only the Setup cell: you can start here.

D/P is the market's price against its fundamentals. A long-short factor has one too: how cheap the stocks it buys are
against the stocks it sells,

$$\text{VS}_t = \log \frac{B}{M}\Big|_{\text{long},\,t} - \log \frac{B}{M}\Big|_{\text{short},\,t}.$$

`four_factors` has four factors from Ken French's 2×3 sorts: each one's monthly return (`HML`, `SMB`, `RMW`, `CMA`) and
the book-to-market of its long and short legs (`HML_bm_long`, `HML_bm_short`, and so on). HML and SMB start in 1926, RMW
and CMA in 1963. Book is from the last fiscal year; the price is from last December.

**Your turn.** Build HML's value spread and call it `vs_hml`. `np.log` takes logs.

In [ ]:
# === YOUR TURN ===

vs_hml = ____

In [ ]:
#@title 🔒 Check: HML's value spread
hml_spread = np.log(four_factors['HML_bm_long']) - np.log(four_factors['HML_bm_short'])

try:
    print("yours matches" if np.allclose(vs_hml, hml_spread) else "yours differs: compare with the plot below", "\n")
except Exception as e:
    print("yours: not built yet:", type(e).__name__, "\n")

hml_next_5_years = four_factors['HML'].rolling(60).sum().shift(-60) / 5     # HML's return over the next five years, a year
ax = hml_spread.plot(label="HML's value spread", legend=True)
hml_next_5_years.plot(ax=ax, secondary_y=True, label='HML over the next five years, a year', legend=True)
plt.show()

print(f"median {hml_spread.median():.2f}, now {hml_spread.iloc[-1]:.2f} ({hml_spread.index[-1]:%Y-%m}); "
      f"correlation with the next five years: {hml_spread.corr(hml_next_5_years):.2f}")

Now sections 1 to 3 again, in a loop over the four factors: the in-sample regression on next month's return, the
out-of-sample forecasts from 1980, and the timing strategy against buy and hold with the same average weight. The
variance is held at its value before 1980.

In [ ]:
table = {}
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for factor, ax in zip(['HML', 'SMB', 'RMW', 'CMA'], axes.flat):
    data = four_factors[[factor]].copy()
    data['spread']   = np.log(four_factors[f'{factor}_bm_long']) - np.log(four_factors[f'{factor}_bm_short'])   # this factor's value spread
    data['next_ret'] = data[factor].shift(-1)                 # next month's return, on this month's row
    data = data.dropna()

    # in sample, as in section 1
    fit = sm.OLS(data['next_ret'], sm.add_constant(data['spread'])).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

    # out of sample from 1980, as in section 2
    forecasts = out_of_sample(data, 'spread', '1980')

    # the timing strategy against buy and hold, as in section 3
    variance = four_factors.loc[:'1979', factor].var()
    forecasts['weight']       = forecasts['forecast'] / (3 * variance)
    forecasts['timing']       = forecasts['weight'] * forecasts['ret']
    forecasts['buy and hold'] = forecasts['weight'].mean() * forecasts['ret']
    (1 + forecasts[['timing', 'buy and hold']]).cumprod().plot(ax=ax, logy=True, title=factor)

    table[factor] = {'from': data.index[0].year, 'HAC t': fit.tvalues['spread'], 'R2 in sample': fit.rsquared,
                     'R2 out of sample': r2_out_of_sample(forecasts),
                     'Sharpe, timing': sharpe(forecasts['timing']), 'Sharpe, buy and hold': sharpe(forecasts['buy and hold']),
                     'average weight': forecasts['weight'].mean(), 'months short': np.mean(forecasts['weight'] < 0)}
plt.suptitle('cumulative return: timing on the value spread, and buy and hold with the same average weight')
plt.tight_layout(); plt.show()

print(pd.DataFrame.from_dict(table, orient='index').round(3).to_string())

- Which factors beat the historical average out of sample? Set them against D/P's $R^2$ in section 2.
- Where does timing pull ahead of buy and hold, and when? How often is that strategy short, and would you run it?
- The price in the book-to-market is last December's, not today's. What has the factor done since December, and why
  might that help the forecast?

---

## 5. Timing the six factors: 24 tries

> 🧩 **Stands alone.** Needs only the Setup cell: you can start here.

Four simple signals for each factor: its own return last month, over the last 12 months, three to five years ago, and its
own volatility. Run the forecasting regression on 1980–2002 and again on 2003–2026. A real signal should work in both.

In [ ]:
daily = pd.read_parquet(f"{BASE}/ff6_daily_1963.parquet")[six_factors.columns]   # the six factors, daily
month = daily.index + pd.offsets.MonthEnd(0)                                     # the month each day belongs to
volatility = daily.groupby(month).apply(lambda days: np.sqrt(((days - days.mean()) ** 2).sum()))   # each factor's volatility, month by month
volatility.index.name = 'date'

rows = []
for factor in six_factors.columns:
    data = six_factors[[factor]].rename(columns={factor: 'ret'})
    data = data.merge(volatility[[factor]].rename(columns={factor: 'own volatility'}), on='date', how='left')
    data['last month']     = data['ret']
    data['last 12 months'] = data['ret'].rolling(12).sum()
    data['3-5 years ago']  = data['ret'].shift(12).rolling(48).sum()
    data['next_ret']       = data['ret'].shift(-1)            # next month's return, on this month's row

    for signal in ['last month', 'last 12 months', '3-5 years ago', 'own volatility']:
        row = {'factor': factor, 'signal': signal}
        for half, first, last in [('t, 1980-2002', '1980', '2002'), ('t, 2003-2026', '2003', '2026')]:
            period = data.loc[first:last, [signal, 'next_ret']].dropna()
            row[half] = sm.OLS(period['next_ret'], sm.add_constant(period[signal])).fit().tvalues[signal]
        rows.append(row)
tstats = pd.DataFrame(rows)
print(tstats.round(2).sort_values('t, 1980-2002', ascending=False).to_string(index=False))

plt.scatter(tstats['t, 1980-2002'], tstats['t, 2003-2026'])
plt.axhline(0, color='k', lw=0.5); plt.axvline(0, color='k', lw=0.5)
plt.xlabel('t, 1980-2002'); plt.ylabel('t, 2003-2026'); plt.show()

- How many of the 24 have $|t| > 2$ in the first half? How many would you expect by luck alone?
- Take the five best from the first half. How many still work in the second?
- Which signal works for several factors in both halves? It has a name: factor momentum.

---

## 6. Pick a predictor

> 🧩 **Stands alone.** Needs only the Setup cell: you can start here.

The slides list eleven signals people use to forecast the market. `predictors` has ten of them, one column each, on the
same months as `market`. Forward P/E is missing: it needs analysts' forecasts, which we do not have.

| column | what it is | from |
|---|---|---|
| `D/P` | dividends over the last 12 months, over price | 1927 |
| `E/P` | S&P 500 earnings over the last 12 months, over price | 1927 |
| `1/CAPE` | ten years of real earnings over the real price: Shiller's CAPE, as a yield | 1927 |
| `book-to-market` | book value over market value, Dow Jones Industrials | 1927 |
| `short rate` | 3-month T-bill yield | 1927 |
| `term spread` | long-term government bond yield minus the T-bill | 1927 |
| `default spread` | Baa minus Aaa corporate bond yields | 1927 |
| `realized variance` | sum of squared daily S&P 500 returns in the month | 1927 |
| `net issuance` | net equity issued over the last 12 months, over market value | 1927 |
| `VIX` | option-implied volatility of the S&P 500 | 1990 |

The valuation ratios are all yields, like D/P: a high value means a cheap market.

Set `PREDICTOR` to a column name and run the cell. It does sections 1 to 3 on that signal: the in-sample regression, the
out-of-sample forecasts from `START`, and the timing strategy against buy and hold.

In [ ]:
PREDICTOR = 'E/P'      # one of: 'D/P', 'E/P', '1/CAPE', 'book-to-market', 'short rate', 'term spread', 'default spread',
                       #         'realized variance', 'net issuance', 'VIX'
START     = '1950'     # the first month to forecast

data  = predictors[[PREDICTOR]].merge(market[['next_ret', 'next_log_ret']], on='date').dropna()   # this month's signal, next month's return
start = max(pd.Timestamp(START), data.index[0] + pd.DateOffset(years=10))      # at least ten years of data before the first forecast

# in sample, as in section 1
fit = sm.OLS(data['next_log_ret'], sm.add_constant(data[PREDICTOR])).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

# out of sample, as in section 2
forecasts = out_of_sample(data, PREDICTOR, start)

# the timing strategy against buy and hold, as in section 3, with the variance held at its value before the first forecast
variance = market.loc[market.index < start, 'ret'].var()
forecasts['weight']       = forecasts['forecast'] / (3 * variance)
forecasts['timing']       = forecasts['weight'] * forecasts['ret']
forecasts['buy and hold'] = forecasts['weight'].mean() * forecasts['ret']

print(f"{PREDICTOR}, {data.index[0]:%Y-%m} to {data.index[-1]:%Y-%m}")
print(f"in sample:     b {fit.params[PREDICTOR]:.3f}, HAC t {fit.tvalues[PREDICTOR]:.2f}, R2 {fit.rsquared:.2%}")
print(f"out of sample: R2 {r2_out_of_sample(forecasts):.2%}, with forecasts from {start:%Y-%m}")
print(f"Sharpe ratio:  timing {sharpe(forecasts['timing']):.2f}, buy and hold {sharpe(forecasts['buy and hold']):.2f}; "
      f"average weight {forecasts['weight'].mean():.2f}, short in {np.mean(forecasts['weight'] < 0):.0%} of months")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
data[PREDICTOR].plot(ax=axes[0], title=PREDICTOR)
(1 + forecasts[['timing', 'buy and hold']]).cumprod().plot(ax=axes[1], logy=True, title='cumulative return: timing, and buy and hold with the same average weight')
plt.show()

- Try `E/P`, the strongest in sample, then `short rate`. Which beats the historical average out of sample?
- Does it beat buy and hold: on the plot, and on the Sharpe ratio? If the two disagree, which do you believe?
- Try `VIX`. Its forecasts start in 2000, after ten years of data. Is that enough to estimate a slope?
- Pick a signal and set `START = '1980'`. Does the verdict change?
- Every one of these signals was published because it forecast well in some sample. Why might that make it look
  better in sample than it will do from here on?

The cell below runs the same three steps on all ten at once: the table in the slides.

In [ ]:
#@title All ten at once: the table in the slides
table = {}
for name in predictors.columns:
    data  = predictors[[name]].merge(market[['next_ret', 'next_log_ret']], on='date').dropna()
    start = max(pd.Timestamp('1950'), data.index[0] + pd.DateOffset(years=10))

    fit       = sm.OLS(data['next_log_ret'], sm.add_constant(data[name])).fit(cov_type='HAC', cov_kwds={'maxlags': 1})
    forecasts = out_of_sample(data, name, start)
    forecasts['timing'] = forecasts['forecast'] / (3 * market.loc[:'1949', 'ret'].var()) * forecasts['ret']

    table[name] = {'from': data.index[0].year, 'HAC t': fit.tvalues[name], 'forecasts from': start.year,
                   'R2 out of sample': r2_out_of_sample(forecasts), 'R2, 2000-2024': r2_out_of_sample(forecasts.loc['2000':]),
                   'Sharpe, timing': sharpe(forecasts['timing']), 'Sharpe, buy and hold': sharpe(forecasts['ret'])}

print(pd.DataFrame.from_dict(table, orient='index').round(4).to_string())

---

## ⏸ Back to the slides

For the takeaways. The challenge below is homework.

---

## 🎯 Challenge: start later

> 🧩 **Stands alone.** Needs only the Setup cell.

*Homework — due before the next class. It should take ten minutes.*

Section 2 started forecasting in 1950. Suppose you had only started in **1980**, the year the value spreads in section 4
start. Redo section 2 with forecasts from 1980 and everything else the same: change `'1950'` to `'1980'` in the loop, or
use `out_of_sample()` and `r2_out_of_sample()` from Setup. Report the out-of-sample $R^2$.

Then, in at most three sentences: would you trade D/P on this evidence? Set it against the value spreads in section 4.

> **📌 Required variable names:**
> ```python
> oos_r2_1980 = ____   # out-of-sample R2 with forecasts from 1980, a decimal (-0.01 means -1%)
> MEMO = """...""" # at most three sentences
> ```

In [ ]:
# Your work here


oos_r2_1980 = ____

print(f"out of sample R2 from 1980: {oos_r2_1980:.2%}")

In [ ]:
MEMO = """
Write your answer here. Don't delete the surrounding triple quotes.
"""
print(MEMO)

---

## 📤 Submission

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ['oos_r2_1980', 'MEMO']
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")

payload = {
    "assignment": "L15_ConditionalStrategiesII_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")